<div style="text-align: center; padding: 40px 0; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 10px; margin-bottom: 30px;">
  <h1 style="font-size: 2.5em; margin: 0; font-weight: 700;">Orchestrating Autonomy</h1>
  <h2 style="font-size: 1.5em; margin: 20px 0; font-weight: 400; opacity: 0.95;">Designing Multi-Agent AI Systems</h2>
  <hr style="width: 60%; margin: 25px auto; border: 1px solid rgba(255,255,255,0.3);">
  <p style="font-size: 1.2em; margin: 15px 0; font-weight: 300;">Pranjal Joshi</p>
  <p style="font-size: 0.9em; opacity: 0.8; margin-top: 20px;">Interactive Live Demo</p>
</div>

## 🎯 Demo Overview

In this interactive demo, we'll build a **production-ready multi-agent system** that orchestrates multiple AI agents to collaborate on complex tasks. You'll see:

- 🤖 **Real AI agents** powered by OpenAI's GPT models
- 🔄 **Live orchestration** with sequential, parallel, and hierarchical patterns
- 📊 **Interactive visualizations** of agent communication and decision-making
- 🏗️ **Architecture diagrams** showing system design
- 💼 **Real-world example**: A multi-agent research and writing team

---

## 📦 Setup & Installation

First, let's set up our environment with all necessary dependencies.

In [ ]:
# Install required packages (run once)
!pip install -q openai langgraph langchain langchain-openai python-dotenv
!pip install -q plotly networkx matplotlib ipywidgets

print("✅ All packages installed successfully!")

In [ ]:
# Imports
import os
import json
import time
from datetime import datetime
from typing import Annotated, TypedDict, List, Dict, Any, Literal
from IPython.display import display, HTML, Markdown, Image
import warnings
warnings.filterwarnings('ignore')

# LangGraph & LangChain
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Visualization
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ All imports successful!")

### 🔑 API Key Configuration

Set your OpenAI API key (get one at [platform.openai.com](https://platform.openai.com)):

In [ ]:
# Option 1: Set directly (for demo)
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Option 2: Load from .env file (recommended for production)
# from dotenv import load_dotenv
# load_dotenv()

# Verify API key is set
if os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY") != "your-api-key-here":
    print("✅ OpenAI API key configured")
else:
    print("⚠️  Please set your OPENAI_API_KEY above")

---

## 🏗️ Architecture Overview

Let's visualize the multi-agent system architecture we're building:

In [ ]:
# Display Mermaid diagram using HTML
mermaid_diagram = """
<div class="mermaid">
graph TB
    User[👤 User Request] --> Orchestrator[🎯 Orchestrator Agent]
    
    Orchestrator --> Researcher[🔍 Researcher Agent]
    Orchestrator --> Planner[📋 Planner Agent]
    Orchestrator --> Writer[✍️ Writer Agent]
    Orchestrator --> Critic[🔎 Critic Agent]
    
    Researcher -->|Findings| Orchestrator
    Planner -->|Outline| Orchestrator
    Writer -->|Draft| Orchestrator
    Critic -->|Feedback| Orchestrator
    
    Orchestrator -->|Final Output| Result[📄 Final Result]
    
    style User fill:#667eea,stroke:#333,stroke-width:3px,color:#fff
    style Orchestrator fill:#764ba2,stroke:#333,stroke-width:3px,color:#fff
    style Researcher fill:#f093fb,stroke:#333,stroke-width:2px
    style Planner fill:#4facfe,stroke:#333,stroke-width:2px
    style Writer fill:#43e97b,stroke:#333,stroke-width:2px
    style Critic fill:#fa709a,stroke:#333,stroke-width:2px
    style Result fill:#feca57,stroke:#333,stroke-width:3px
</div>

<script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
<script>
    mermaid.initialize({startOnLoad:true, theme: 'default'});
</script>
"""

display(HTML(mermaid_diagram))

### Agent Roles

| Agent | Role | Capabilities |
|-------|------|-------------|
| 🎯 **Orchestrator** | Coordinates workflow | Task delegation, result aggregation, quality control |
| 🔍 **Researcher** | Information gathering | Web research, fact-finding, data collection |
| 📋 **Planner** | Strategic planning | Task breakdown, outline creation, structure |
| ✍️ **Writer** | Content creation | Drafting, composition, synthesis |
| 🔎 **Critic** | Quality assurance | Review, feedback, improvement suggestions |

---

## 🎨 Visualization Utilities

Let's create helper functions for beautiful, interactive visualizations:

In [ ]:
class AgentVisualizer:
    """Interactive visualization suite for multi-agent systems."""
    
    COLORS = {
        'Orchestrator': '#764ba2',
        'Researcher': '#f093fb',
        'Planner': '#4facfe',
        'Writer': '#43e97b',
        'Critic': '#fa709a'
    }
    
    @staticmethod
    def visualize_communication_flow(messages: List[Dict]):
        """Create an interactive Sankey diagram of agent communications."""
        agents = list(set([msg['from'] for msg in messages] + [msg['to'] for msg in messages]))
        agent_indices = {agent: i for i, agent in enumerate(agents)}
        
        source = [agent_indices[msg['from']] for msg in messages]
        target = [agent_indices[msg['to']] for msg in messages]
        value = [1] * len(messages)
        
        colors = [AgentVisualizer.COLORS.get(agent, '#95a5a6') for agent in agents]
        
        fig = go.Figure(data=[go.Sankey(
            node=dict(
                pad=20,
                thickness=25,
                line=dict(color="black", width=0.5),
                label=agents,
                color=colors,
                customdata=[f"Agent: {agent}" for agent in agents],
                hovertemplate='%{customdata}<br>Messages: %{value}<extra></extra>'
            ),
            link=dict(
                source=source,
                target=target,
                value=value,
                color='rgba(118, 75, 162, 0.3)'
            )
        )])
        
        fig.update_layout(
            title="Agent Communication Flow",
            font=dict(size=12, family="Arial"),
            height=500,
            plot_bgcolor='white'
        )
        
        fig.show()
    
    @staticmethod
    def visualize_execution_timeline(events: List[Dict]):
        """Create an interactive Gantt chart of agent execution."""
        fig = go.Figure()
        
        for i, event in enumerate(events):
            agent = event['agent']
            color = AgentVisualizer.COLORS.get(agent, '#95a5a6')
            
            fig.add_trace(go.Scatter(
                x=[event['start'], event['end']],
                y=[agent, agent],
                mode='lines+markers',
                name=event['task'],
                line=dict(width=15, color=color),
                marker=dict(size=10, color=color),
                hovertemplate=f"<b>{agent}</b><br>" +
                             f"Task: {event['task']}<br>" +
                             f"Duration: {event['end'] - event['start']:.2f}s<extra></extra>"
            ))
        
        fig.update_layout(
            title="Agent Execution Timeline",
            xaxis_title="Time (seconds)",
            yaxis_title="Agent",
            height=400,
            showlegend=True,
            hovermode='closest',
            plot_bgcolor='white',
            font=dict(size=12)
        )
        
        fig.show()
    
    @staticmethod
    def visualize_agent_network(agents: List[str], connections: List[tuple]):
        """Create a network graph of agent relationships."""
        G = nx.DiGraph()
        G.add_nodes_from(agents)
        G.add_edges_from(connections)
        
        plt.figure(figsize=(14, 10))
        pos = nx.spring_layout(G, k=3, iterations=50, seed=42)
        
        colors = [AgentVisualizer.COLORS.get(node, '#95a5a6') for node in G.nodes()]
        
        nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=4000, alpha=0.9, edgecolors='black', linewidths=2)
        nx.draw_networkx_labels(G, pos, font_size=11, font_weight='bold', font_color='white')
        nx.draw_networkx_edges(
            G, pos,
            edge_color='#34495e',
            arrows=True,
            arrowsize=25,
            arrowstyle='->',
            width=2.5,
            connectionstyle='arc3,rad=0.1',
            alpha=0.7
        )
        
        plt.title("Multi-Agent System Network", fontsize=16, fontweight='bold', pad=20)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def display_agent_activity(agent_name: str, activity: str, status: str = "processing"):
        """Display real-time agent activity with nice formatting."""
        status_icons = {
            "processing": "⚙️",
            "complete": "✅",
            "error": "❌"
        }
        
        color = AgentVisualizer.COLORS.get(agent_name, '#95a5a6')
        icon = status_icons.get(status, "🔄")
        
        html = f"""
        <div style="
            padding: 15px;
            margin: 10px 0;
            background: linear-gradient(135deg, {color}22 0%, {color}11 100%);
            border-left: 4px solid {color};
            border-radius: 5px;
            font-family: 'Courier New', monospace;
        ">
            <strong style="color: {color}; font-size: 1.1em;">{icon} {agent_name}</strong>
            <p style="margin: 8px 0 0 0; color: #2c3e50;">{activity}</p>
        </div>
        """
        display(HTML(html))

print("✅ Visualization utilities loaded")

---

## 🤖 Building the Multi-Agent System

Now let's build our production-ready multi-agent system using LangGraph and OpenAI.

### Step 1: Define the Shared State

In [ ]:
class ResearchState(TypedDict):
    """Shared state across all agents."""
    messages: Annotated[List, add_messages]
    topic: str
    research_findings: str
    outline: str
    draft: str
    feedback: str
    final_output: str
    next_agent: str
    iteration: int
    execution_log: List[Dict[str, Any]]

print("✅ State schema defined")

### Step 2: Initialize the LLM

In [ ]:
# Initialize OpenAI model
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Fast and cost-effective
    temperature=0.7,
    max_tokens=2000
)

print("✅ LLM initialized (gpt-4o-mini)")

### Step 3: Create Specialized Agents

In [ ]:
class MultiAgentSystem:
    """Production multi-agent orchestration system."""
    
    def __init__(self, llm):
        self.llm = llm
        self.visualizer = AgentVisualizer()
        self.start_time = None
    
    def researcher_agent(self, state: ResearchState) -> ResearchState:
        """Agent specialized in research and information gathering."""
        start = time.time()
        self.visualizer.display_agent_activity(
            "Researcher",
            f"Researching topic: {state['topic']}",
            "processing"
        )
        
        prompt = f"""
You are a Research Agent specialized in gathering comprehensive information.

Topic: {state['topic']}

Your task:
1. Identify key concepts and themes
2. Outline important facts and statistics
3. Note relevant examples and case studies
4. Suggest credible sources

Provide concise, well-structured research findings.
"""
        
        response = self.llm.invoke([HumanMessage(content=prompt)])
        state["research_findings"] = response.content
        state["messages"].append(AIMessage(content=f"[Researcher] {response.content[:200]}..."))
        
        duration = time.time() - start
        state["execution_log"].append({
            'agent': 'Researcher',
            'task': 'Research topic',
            'start': start - self.start_time,
            'end': time.time() - self.start_time
        })
        
        self.visualizer.display_agent_activity(
            "Researcher",
            f"Research complete ({duration:.1f}s)",
            "complete"
        )
        
        state["next_agent"] = "planner"
        return state
    
    def planner_agent(self, state: ResearchState) -> ResearchState:
        """Agent specialized in planning and structuring."""
        start = time.time()
        self.visualizer.display_agent_activity(
            "Planner",
            "Creating structured outline",
            "processing"
        )
        
        prompt = f"""
You are a Planning Agent specialized in creating clear, logical structures.

Topic: {state['topic']}
Research Findings:
{state['research_findings']}

Your task:
1. Create a clear outline with main sections
2. Ensure logical flow of ideas
3. Identify key points for each section
4. Suggest a compelling narrative arc

Provide a detailed outline.
"""
        
        response = self.llm.invoke([HumanMessage(content=prompt)])
        state["outline"] = response.content
        state["messages"].append(AIMessage(content=f"[Planner] {response.content[:200]}..."))
        
        duration = time.time() - start
        state["execution_log"].append({
            'agent': 'Planner',
            'task': 'Create outline',
            'start': start - self.start_time,
            'end': time.time() - self.start_time
        })
        
        self.visualizer.display_agent_activity(
            "Planner",
            f"Outline complete ({duration:.1f}s)",
            "complete"
        )
        
        state["next_agent"] = "writer"
        return state
    
    def writer_agent(self, state: ResearchState) -> ResearchState:
        """Agent specialized in content creation."""
        start = time.time()
        self.visualizer.display_agent_activity(
            "Writer",
            "Drafting content based on outline",
            "processing"
        )
        
        prompt = f"""
You are a Writing Agent specialized in creating engaging, informative content.

Topic: {state['topic']}
Research:
{state['research_findings'][:500]}...

Outline:
{state['outline']}

Your task:
1. Write clear, engaging prose
2. Follow the provided outline
3. Incorporate research findings naturally
4. Maintain consistent tone and style

Write a comprehensive draft (400-600 words).
"""
        
        response = self.llm.invoke([HumanMessage(content=prompt)])
        state["draft"] = response.content
        state["messages"].append(AIMessage(content=f"[Writer] {response.content[:200]}..."))
        
        duration = time.time() - start
        state["execution_log"].append({
            'agent': 'Writer',
            'task': 'Write draft',
            'start': start - self.start_time,
            'end': time.time() - self.start_time
        })
        
        self.visualizer.display_agent_activity(
            "Writer",
            f"Draft complete ({duration:.1f}s)",
            "complete"
        )
        
        state["next_agent"] = "critic"
        return state
    
    def critic_agent(self, state: ResearchState) -> ResearchState:
        """Agent specialized in review and quality control."""
        start = time.time()
        self.visualizer.display_agent_activity(
            "Critic",
            "Reviewing draft for quality and accuracy",
            "processing"
        )
        
        prompt = f"""
You are a Critic Agent specialized in quality assurance and improvement.

Draft to review:
{state['draft']}

Your task:
1. Evaluate clarity, coherence, and completeness
2. Check factual accuracy against research
3. Identify areas for improvement
4. Provide specific, actionable feedback
5. Give an overall quality score (1-10)

If score >= 8, approve for publication.
If score < 8, suggest specific improvements.
"""
        
        response = self.llm.invoke([HumanMessage(content=prompt)])
        state["feedback"] = response.content
        state["messages"].append(AIMessage(content=f"[Critic] {response.content[:200]}..."))
        
        duration = time.time() - start
        state["execution_log"].append({
            'agent': 'Critic',
            'task': 'Review draft',
            'start': start - self.start_time,
            'end': time.time() - self.start_time
        })
        
        # Simple check: if feedback contains high score, finalize
        # In production, you'd parse the score more carefully
        if any(word in response.content.lower() for word in ["approve", "publish", "excellent", "score: 8", "score: 9", "score: 10", "8/10", "9/10", "10/10"]):
            state["final_output"] = state["draft"]
            state["next_agent"] = "FINISH"
            self.visualizer.display_agent_activity(
                "Critic",
                f"✨ Draft approved for publication ({duration:.1f}s)",
                "complete"
            )
        else:
            state["iteration"] += 1
            if state["iteration"] < 2:  # Max 2 iterations for demo
                state["next_agent"] = "writer"
                self.visualizer.display_agent_activity(
                    "Critic",
                    f"Improvements needed - sending back to Writer ({duration:.1f}s)",
                    "complete"
                )
            else:
                state["final_output"] = state["draft"]
                state["next_agent"] = "FINISH"
                self.visualizer.display_agent_activity(
                    "Critic",
                    f"Max iterations reached - finalizing ({duration:.1f}s)",
                    "complete"
                )
        
        return state
    
    def orchestrator_router(self, state: ResearchState) -> Literal["researcher", "planner", "writer", "critic", "__end__"]:
        """Route to the next agent based on current state."""
        next_agent = state.get("next_agent", "researcher")
        
        if next_agent == "FINISH":
            return "__end__"
        return next_agent
    
    def build_graph(self):
        """Build the LangGraph workflow."""
        workflow = StateGraph(ResearchState)
        
        # Add agent nodes
        workflow.add_node("researcher", self.researcher_agent)
        workflow.add_node("planner", self.planner_agent)
        workflow.add_node("writer", self.writer_agent)
        workflow.add_node("critic", self.critic_agent)
        
        # Set entry point
        workflow.set_entry_point("researcher")
        
        # Add conditional edges for dynamic routing
        workflow.add_conditional_edges(
            "researcher",
            self.orchestrator_router,
            {"planner": "planner"}
        )
        workflow.add_conditional_edges(
            "planner",
            self.orchestrator_router,
            {"writer": "writer"}
        )
        workflow.add_conditional_edges(
            "writer",
            self.orchestrator_router,
            {"critic": "critic"}
        )
        workflow.add_conditional_edges(
            "critic",
            self.orchestrator_router,
            {
                "writer": "writer",
                "__end__": END
            }
        )
        
        return workflow.compile()

print("✅ Multi-agent system class defined")

---

## 🚀 Live Demonstration

Let's run our multi-agent system on a real task!

### Initialize the System

In [ ]:
# Create the multi-agent system
mas = MultiAgentSystem(llm)
app = mas.build_graph()

print("✅ Multi-agent workflow compiled and ready!")

# Visualize the agent network
agents = ["Orchestrator", "Researcher", "Planner", "Writer", "Critic"]
connections = [
    ("Orchestrator", "Researcher"),
    ("Orchestrator", "Planner"),
    ("Orchestrator", "Writer"),
    ("Orchestrator", "Critic"),
    ("Researcher", "Planner"),
    ("Planner", "Writer"),
    ("Writer", "Critic"),
    ("Critic", "Writer"),  # Feedback loop
]

mas.visualizer.visualize_agent_network(agents, connections)

### Execute the Workflow

In [ ]:
# Define the task
topic = "The Future of Multi-Agent AI Systems in Enterprise Applications"

display(HTML(f"""
<div style="
    padding: 25px;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
    border-radius: 10px;
    text-align: center;
    margin: 20px 0;
">
    <h2 style="margin: 0; font-size: 1.8em;">🎯 Task</h2>
    <p style="margin: 15px 0 0 0; font-size: 1.2em; font-weight: 300;">{topic}</p>
</div>
"""))

# Initialize state
initial_state = {
    "messages": [HumanMessage(content=f"Write a comprehensive article about: {topic}")],
    "topic": topic,
    "research_findings": "",
    "outline": "",
    "draft": "",
    "feedback": "",
    "final_output": "",
    "next_agent": "researcher",
    "iteration": 0,
    "execution_log": []
}

# Run the workflow
print("\n" + "="*80)
print("🚀 STARTING MULTI-AGENT EXECUTION")
print("="*80 + "\n")

mas.start_time = time.time()
final_state = app.invoke(initial_state)

total_time = time.time() - mas.start_time

print("\n" + "="*80)
print(f"✅ EXECUTION COMPLETE ({total_time:.1f}s total)")
print("="*80)

---

## 📊 Visualize Results

Let's analyze the agent interactions and performance:

### Communication Flow

In [ ]:
# Extract communication messages
messages = [
    {'from': 'User', 'to': 'Researcher'},
    {'from': 'Researcher', 'to': 'Planner'},
    {'from': 'Planner', 'to': 'Writer'},
    {'from': 'Writer', 'to': 'Critic'},
]

# Add feedback loop if there was iteration
if final_state['iteration'] > 0:
    messages.append({'from': 'Critic', 'to': 'Writer'})
    messages.append({'from': 'Writer', 'to': 'Critic'})

messages.append({'from': 'Critic', 'to': 'User'})

mas.visualizer.visualize_communication_flow(messages)

### Execution Timeline

In [ ]:
mas.visualizer.visualize_execution_timeline(final_state['execution_log'])

### Performance Metrics

In [ ]:
# Calculate metrics
agent_times = {}
for event in final_state['execution_log']:
    agent = event['agent']
    duration = event['end'] - event['start']
    if agent not in agent_times:
        agent_times[agent] = 0
    agent_times[agent] += duration

# Create bar chart
fig = go.Figure()

agents_list = list(agent_times.keys())
times_list = list(agent_times.values())
colors_list = [AgentVisualizer.COLORS.get(agent, '#95a5a6') for agent in agents_list]

fig.add_trace(go.Bar(
    x=agents_list,
    y=times_list,
    marker_color=colors_list,
    text=[f"{t:.2f}s" for t in times_list],
    textposition='auto',
))

fig.update_layout(
    title="Agent Execution Time",
    xaxis_title="Agent",
    yaxis_title="Time (seconds)",
    height=400,
    showlegend=False,
    plot_bgcolor='white',
    font=dict(size=12)
)

fig.show()

# Summary stats
display(HTML(f"""
<div style="padding: 20px; background: #f8f9fa; border-radius: 10px; margin: 20px 0;">
    <h3>📈 Performance Summary</h3>
    <ul style="font-size: 1.1em; line-height: 1.8;">
        <li><strong>Total Execution Time:</strong> {total_time:.2f} seconds</li>
        <li><strong>Agents Activated:</strong> {len(agent_times)}</li>
        <li><strong>Total Messages:</strong> {len(final_state['messages'])}</li>
        <li><strong>Iterations:</strong> {final_state['iteration'] + 1}</li>
        <li><strong>Output Length:</strong> {len(final_state['final_output'])} characters</li>
    </ul>
</div>
"""))

---

## 📄 Final Output

Here's the final article produced by our multi-agent system:

In [ ]:
display(HTML(f"""
<div style="
    padding: 40px;
    background: white;
    border: 2px solid #e0e0e0;
    border-radius: 10px;
    margin: 20px 0;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
">
    <h2 style="color: #2c3e50; border-bottom: 3px solid #667eea; padding-bottom: 15px;">
        {topic}
    </h2>
    <div style="
        margin-top: 25px;
        line-height: 1.8;
        color: #34495e;
        font-size: 1.05em;
        white-space: pre-wrap;
    ">
        {final_state['final_output']}
    </div>
    <hr style="margin: 30px 0; border: 1px solid #e0e0e0;">
    <p style="color: #7f8c8d; font-style: italic; text-align: center;">
        ✨ Generated by Multi-Agent AI System | Orchestrated by {len(agent_times)} specialized agents
    </p>
</div>
"""))

### Agent Outputs (Detailed View)

In [ ]:
# Show individual agent contributions
from IPython.display import display, HTML

def show_agent_output(agent_name: str, output: str, color: str):
    display(HTML(f"""
    <div style="margin: 20px 0;">
        <details>
            <summary style="
                padding: 15px;
                background: {color}22;
                border-left: 4px solid {color};
                cursor: pointer;
                font-size: 1.1em;
                font-weight: bold;
                border-radius: 5px;
            ">
                🔍 {agent_name} Output (click to expand)
            </summary>
            <div style="
                padding: 20px;
                background: #f8f9fa;
                border: 1px solid #dee2e6;
                border-top: none;
                white-space: pre-wrap;
                line-height: 1.6;
            ">
                {output}
            </div>
        </details>
    </div>
    """))

show_agent_output("Researcher", final_state['research_findings'], '#f093fb')
show_agent_output("Planner", final_state['outline'], '#4facfe')
show_agent_output("Writer", final_state['draft'], '#43e97b')
show_agent_output("Critic", final_state['feedback'], '#fa709a')

---

## 🎨 System Architecture Patterns

Let's visualize the different orchestration patterns used in this system:

In [ ]:
# Create pattern comparison visualization
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Sequential Pattern', 'Feedback Loop', 'Hierarchical Pattern'),
    specs=[[{'type': 'sankey'}, {'type': 'sankey'}, {'type': 'sankey'}]]
)

# Pattern 1: Sequential
fig.add_trace(go.Sankey(
    node=dict(label=["Researcher", "Planner", "Writer", "Output"], color="#667eea"),
    link=dict(source=[0, 1, 2], target=[1, 2, 3], value=[1, 1, 1])
), row=1, col=1)

# Pattern 2: Feedback Loop
fig.add_trace(go.Sankey(
    node=dict(label=["Writer", "Critic", "Writer V2", "Output"], color="#764ba2"),
    link=dict(source=[0, 1, 2], target=[1, 2, 3], value=[1, 1, 1])
), row=1, col=2)

# Pattern 3: Hierarchical
fig.add_trace(go.Sankey(
    node=dict(label=["Orchestrator", "Researcher", "Planner", "Writer", "Critic"], color="#f093fb"),
    link=dict(source=[0, 0, 0, 0], target=[1, 2, 3, 4], value=[1, 1, 1, 1])
), row=1, col=3)

fig.update_layout(height=400, showlegend=False, title_text="Multi-Agent Orchestration Patterns")
fig.show()

---

## 💡 Key Takeaways

<div style="padding: 30px; background: linear-gradient(135deg, #667eea22 0%, #764ba222 100%); border-radius: 10px; margin: 20px 0;">

### 🎯 What We Demonstrated

1. **Specialized Agents**: Each agent has a distinct role and capability
   - Researcher: Information gathering
   - Planner: Strategic structuring
   - Writer: Content creation
   - Critic: Quality assurance

2. **Orchestration Patterns**:
   - **Sequential**: Linear workflow (Researcher → Planner → Writer)
   - **Feedback Loop**: Iterative refinement (Writer ↔ Critic)
   - **Conditional Routing**: Dynamic agent selection based on state

3. **State Management**:
   - Shared state across all agents
   - Typed schemas for reliability
   - Execution logging for observability

4. **Production Patterns**:
   - Error handling and timeouts
   - Performance monitoring
   - Iteration limits
   - Rich visualization

### 🚀 Next Steps

- **Add more agents**: Fact-checker, SEO optimizer, translator
- **Implement caching**: Reduce API calls and costs
- **Add human-in-the-loop**: Approval gates for critical decisions
- **Parallel execution**: Run independent agents simultaneously
- **Persistence**: Save and resume workflows
- **Monitoring**: Add metrics, logging, and alerting

</div>

---

## 🧪 Interactive Playground

Try it yourself with a different topic!

In [ ]:
# Change this to any topic you want!
custom_topic = "The Impact of Quantum Computing on Cryptography"

# Run the workflow
custom_state = {
    "messages": [HumanMessage(content=f"Write about: {custom_topic}")],
    "topic": custom_topic,
    "research_findings": "",
    "outline": "",
    "draft": "",
    "feedback": "",
    "final_output": "",
    "next_agent": "researcher",
    "iteration": 0,
    "execution_log": []
}

display(HTML(f"""
<div style="padding: 20px; background: #667eea; color: white; border-radius: 10px; text-align: center;">
    <h3>🎯 New Topic: {custom_topic}</h3>
    <p>Running multi-agent workflow...</p>
</div>
"""))

mas.start_time = time.time()
custom_result = app.invoke(custom_state)

display(Markdown(f"## Final Output\n\n{custom_result['final_output']}"))

---

<div style="text-align: center; padding: 40px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 10px; margin-top: 50px;">
    <h2 style="margin: 0; font-size: 2em;">Thank You! 🎉</h2>
    <p style="margin: 20px 0; font-size: 1.2em; opacity: 0.9;">Questions?</p>
    <hr style="width: 50%; margin: 20px auto; border: 1px solid rgba(255,255,255,0.3);">
    <p style="font-size: 1.1em; margin: 10px 0;">Pranjal Joshi</p>
    <p style="font-size: 0.9em; opacity: 0.8;">Orchestrating Autonomy: Designing Multi-Agent AI Systems</p>
</div>

---

## 📚 Resources

### Frameworks Used
- [LangGraph](https://python.langchain.com/docs/langgraph) - Agent orchestration
- [LangChain](https://python.langchain.com/) - LLM integration
- [OpenAI API](https://platform.openai.com/) - Language models

### Learn More
- [Multi-Agent Systems Research](https://arxiv.org/abs/2309.02427)
- [LangGraph Tutorials](https://github.com/langchain-ai/langgraph/tree/main/examples)
- [Production AI Patterns](https://python.langchain.com/docs/use_cases/agent_workflows)

### Code
- This notebook is available at: [github.com/YOUR_REPO](https://github.com/YOUR_REPO)
- Star ⭐ if you found it helpful!

---

*Built with ❤️ using LangGraph, OpenAI, and Python*